# E-Commerce Sales Performance Analysis

**Portfolio Project — Data Analyst**

## Objective
Analyze e-commerce sales-order data to understand sales trends, product/category performance, order status, fulfilment, shipping preferences, and geographic patterns.

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Jupyter Notebook / Google Colab

### Workflow
Data Understanding → Data Wrangling → Missing-Value Analysis → EDA → Business Questions → Visualization → Insights → Export

>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

p


## 1. Load the Dataset

Load the original CSV and inspect its basic structure.

In [ ]:
df = pd.read_csv("data/raw/datapro.csv")
df.head()


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()


## 2. Data Wrangling

Clean column names, normalize selected text fields, convert dates, and standardize postal-code representation.


In [ ]:
df.columns = [c.strip() for c in df.columns]
df.columns = [" ".join(c.split()) for c in df.columns]


In [ ]:
df.apply(
    lambda x: x.astype(str).str.contains(
        r"(?:^\s+|\s+$|\s{2,})",
        regex=True
    ).sum()
)

In [ ]:
cols = ["SKU", "ship-city", "ship-state"]

for col in cols:
    df[col] = df[col].str.split().str.join(' ')

## 3. Missing-Value Analysis

First quantify missing values. Then investigate whether missingness has a business meaning before choosing a treatment.


In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing %': df.isna().mean() * 100
})

missing

In [ ]:
print("Missing Courier Status — order status:")
print(df.loc[df["Courier Status"].isna(), "Status"].value_counts())

print("\nMissing Amount — order status:")
print(df.loc[df["Amount"].isna(), "Status"].value_counts())


### Missing-value treatment

- `ship-city`, `ship-state`, and `ship-postal-code` have a very small proportion of missing values, so the affected rows are removed.
- `Amount` is required for value-based sales analysis, so rows with missing `Amount` are removed rather than assigning an artificial amount.
- `Courier Status` is treated differently. The investigation shows that the missing values are overwhelmingly associated with cancelled orders, so they are labelled `"Cancelled"` to preserve those records for status analysis.

This is a business-rule-based treatment rather than filling values blindly.


In [ ]:
df["Courier Status"] = df["Courier Status"].fillna("Cancelled")

df = df.dropna(
    subset=["ship-city", "ship-state", "ship-postal-code", "Amount"]
)


In [ ]:
# df = df.drop(columns=['Date'])

## 4. Duplicate and Numeric Validation

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)
df.duplicated().sum()
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

In [ ]:
Validate numeric fields

In [ ]:
print("Quantity summary:")
display(df['Qty'].describe())

print("\nAmount summary:")
display(df['Amount'].describe())

print("\nRows with negative quantity:", (df['Qty'] < 0).sum())
print("Rows with negative amount:", (df['Amount'] < 0).sum())

### Final data-quality check

In [ ]:
print("Final shape:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing"))


## 5. Exploratory Data Analysis

Create a compact profile of orders, products, categories, geography, and cities.


In [ ]:
eda_summary = {
    "Unique Orders": df["Order ID"].nunique(),
    "Unique SKUs": df["SKU"].nunique(),
    "Unique Styles": df["Style"].nunique(),
    "Categories": df["Category"].nunique(),
    "States": df["ship-state"].nunique(),
    "Cities": df["ship-city"].nunique()
}

pd.Series(eda_summary, name="Value")


### Order Status

In [ ]:
status_counts = df["Status"].value_counts()

plt.figure(figsize=(10, 6))
status_counts.plot(kind="bar")
plt.title("Order Status Distribution")
plt.xlabel("Status")
plt.ylabel("Order Lines")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Fulfilment Method

In [ ]:
fulfilment_counts = df["Fulfilment"].value_counts()

plt.figure(figsize=(8, 5))
fulfilment_counts.plot(kind="bar")
plt.title("Orders by Fulfilment Method")
plt.xlabel("Fulfilment")
plt.ylabel("Order Lines")
plt.tight_layout()
plt.show()


### Category Distribution

In [ ]:
category_counts = df["Category"].value_counts()

plt.figure(figsize=(8, 5))
category_counts.plot(kind="bar")
plt.title("Orders by Category")
plt.xlabel("Category")
plt.ylabel("Order Lines")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Size Distribution

In [ ]:
size_counts = df["Size"].value_counts()

plt.figure(figsize=(10, 5))
size_counts.plot(kind="bar")
plt.title("Order Lines by Size")
plt.xlabel("Size")
plt.ylabel("Order Lines")
plt.tight_layout()
plt.show()


### Shipping Service Level

In [ ]:
shipping_counts = df["ship-service-level"].value_counts()

plt.figure(figsize=(8, 5))
shipping_counts.plot(kind="bar")
plt.title("Orders by Shipping Service Level")
plt.xlabel("Shipping Service")
plt.ylabel("Order Lines")
plt.tight_layout()
plt.show()


## 6. Business Question 1 — How does reported order value change over time?

In [ ]:
df["Month"] = df["Date"].dt.month

monthly = df.groupby("Month").agg(
    Order_Lines=("Order ID", "size"),
    Unique_Orders=("Order ID", "nunique"),
    Reported_Order_Value=("Amount", "sum"),
    Delivered_Order_Value=(
        "Amount",
        lambda x: x[df.loc[x.index, "Status"].str.contains("Delivered", na=False)].sum()
    )
).reset_index()

monthly


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly["Month"], monthly["Reported_Order_Value"], marker="o", label="Reported Order Value")
plt.plot(monthly["Month"], monthly["Delivered_Order_Value"], marker="o", label="Delivered Order Value")
plt.title("Monthly Reported vs Delivered Order Value")
plt.xlabel("Month")
plt.ylabel("Amount")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 7. Business Question 2 — Which categories generate the highest reported order value?

In [ ]:
category_sales = df.groupby("Category").agg(
    Reported_Order_Value=("Amount", "sum"),
    Quantity=("Qty", "sum"),
    Unique_Orders=("Order ID", "nunique")
).sort_values("Reported_Order_Value", ascending=False)

category_sales


## 8. Business Question 3 — Which product styles generate the highest reported order value?

In [ ]:
top_styles = df.groupby("Style").agg(
    Reported_Order_Value=("Amount", "sum"),
    Quantity=("Qty", "sum"),
    Unique_Orders=("Order ID", "nunique")
).sort_values("Reported_Order_Value", ascending=False).head(10)

top_styles


In [ ]:
plt.figure(figsize=(10, 6))
top_styles["Reported_Order_Value"].sort_values().plot(kind="barh")
plt.title("Top 10 Styles by Reported Order Value")
plt.xlabel("Reported Order Value")
plt.ylabel("Style")
plt.tight_layout()
plt.show()


## 9. Business Question 4 — Which states have the highest reported order value?

In [ ]:
state_sales = df.groupby("ship-state").agg(
    Reported_Order_Value=("Amount", "sum"),
    Unique_Orders=("Order ID", "nunique")
).sort_values("Reported_Order_Value", ascending=False)

top_states = state_sales.head(10).sort_values("Reported_Order_Value")
top_states


In [ ]:
plt.figure(figsize=(10, 6))
top_states["Reported_Order_Value"].plot(kind="barh")
plt.title("Top 10 States by Reported Order Value")
plt.xlabel("Reported Order Value")
plt.ylabel("State")
plt.tight_layout()
plt.show()


## 10. Business Question 5 — How do fulfilment methods compare?

In [ ]:
fulfilment_analysis = df.groupby("Fulfilment").agg(
    Unique_Orders=("Order ID", "nunique"),
    Quantity=("Qty", "sum"),
    Reported_Order_Value=("Amount", "sum"),
    Delivered_Order_Value=(
        "Amount",
        lambda x: x[df.loc[x.index, "Status"].str.contains("Delivered", na=False)].sum()
    )
).sort_values("Reported_Order_Value", ascending=False)

fulfilment_analysis


## 11. Business Question 6 — What is the order-status mix?

In [ ]:
status_analysis = df.groupby("Status").agg(
    Order_Lines=("Order ID", "size"),
    Unique_Orders=("Order ID", "nunique"),
    Reported_Order_Value=("Amount", "sum")
).sort_values("Unique_Orders", ascending=False)

status_analysis


## 12. Business Question 7 — Which sizes are most common?

In [ ]:
size_analysis = df.groupby("Size").agg(
    Order_Lines=("Order ID", "size"),
    Quantity=("Qty", "sum"),
    Reported_Order_Value=("Amount", "sum")
).sort_values("Order_Lines", ascending=False)

size_analysis


## 13. Additional Analysis — Quantity and Amount

In [ ]:
qty_amount = df[["Qty", "Amount"]].dropna()

plt.figure(figsize=(8, 5))
plt.scatter(qty_amount["Qty"], qty_amount["Amount"], alpha=0.25)
plt.title("Quantity vs Amount")
plt.xlabel("Quantity")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

print("Correlation:", qty_amount["Qty"].corr(qty_amount["Amount"]))


## 14. Key Findings

The cleaned dataset contains **121,146 rows** after the documented cleaning steps.

Selected findings:
- **Set** is the highest reported-value category.
- **JNE3797** is the highest reported-value style.
- **Maharashtra** has the highest reported order value among states.
- **Amazon** has the larger reported order value among fulfilment methods.
- **Shipped** is the most common order status after cleaning.
- The quantity-to-amount correlation is low, showing that higher quantity alone does not strongly explain reported order value in this dataset.

These findings describe the cleaned dataset and should be interpreted as reported order-line value rather than automatically as recognized revenue.


## 15. Business Recommendations

1. Monitor high-value categories and styles for inventory and campaign planning.
2. Compare fulfilment methods using both order volume and delivered-order value.
3. Investigate cancellation patterns and their operational causes.
4. Use state-level demand patterns to support regional inventory planning.
5. Track monthly reported and delivered value separately to avoid overstating realized sales.
6. Continue improving data-quality rules whenever a missing field has a structural business meaning.


## 16. Export Cleaned Dataset

In [ ]:
df.to_csv("data/processed/amazon_sales_cleaned.csv", index=False)
print("Saved: data/processed/amazon_sales_cleaned.csv")
